# EDL Full Experiment (auto-parallel)

10-fold cross-validation comparison of 8 LDL models across 3 datasets
(`SJAFFE`, `SBU_3DFE`, `Human_Gene`).

The pool config is **auto-detected** by `edl_workers.auto_pool_config()`:

- **GPU mode** if ≥ 2 CUDA GPUs are visible to `nvidia-smi` — one worker
  pinned per GPU.
- **CPU mode** otherwise — a small number of fat CPU workers (≈
  `cpu_count // 4`) so TF's intra/inter-op threads don't oversubscribe.

The actual training loop lives in `edl_workers.py` next to this notebook
so loky subprocesses can `import edl_workers` cleanly. TensorFlow is
imported lazily inside each worker after `CUDA_VISIBLE_DEVICES` is pinned.

For each (model, dataset) pair we record six distributional metrics
(`chebyshev`, `clark`, `canberra`, `kl_divergence`, `cosine`, `intersection`)
across 10 folds with a 10% test split per fold, then summarise as
mean ± std.

For the evidential models (`EDL_LDL`, `BEDL_LDL`) and `SNEFY_LDL` we also
report:

- **Mean uncertainty** — average per-sample uncertainty on the test set
  (lower = the model is more confident).
- **Uncertainty calibration (Spearman ρ)** — rank correlation between
  per-sample uncertainty and per-sample KL divergence error.
  Higher = uncertainty tracks error better.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from loky import get_reusable_executor

# Make edl_workers importable whether the kernel started in `demo/` or in the
# project root.
_demo_dir = Path.cwd() if (Path.cwd() / 'edl_workers.py').exists() else Path.cwd() / 'demo'
if str(_demo_dir) not in sys.path:
    sys.path.insert(0, str(_demo_dir))

from pyldl.utils import load_dataset
from edl_workers import (
    init_worker, run_one_fold, auto_pool_config,
    MODEL_NAMES, METRICS,
)


E0000 00:00:1777172304.369549 2149063 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777172304.968469 2149063 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777172307.396307 2149063 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172307.396344 2149063 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172307.396346 2149063 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172307.396348 2149063 computation_placer.cc:177] computation placer already registered. Please check linka

## Configuration

`auto_pool_config()` picks GPU vs CPU mode based on what's actually present.
Override `MODE_OVERRIDE` if you want to force one or the other (useful for
A/B testing GPU-vs-CPU on the same box).


In [2]:
DATASETS = ['SJAFFE', 'SBU_3DFE', 'Movie', 'Music']
N_SPLITS = 10
N_EPOCHS = 100
RANDOM_STATE = 0

# --- Parallel config (auto-detected) -------------------------------------
# Set to None to auto-detect, or pass a dict to override e.g.
#   POOL_CFG = {'mode': 'CPU', 'gpu_ids': [], 'n_workers': 4,
#               'intra_op_threads': 1, 'inter_op_threads': 1}
POOL_CFG = None
# -------------------------------------------------------------------------

if POOL_CFG is None:
    POOL_CFG = auto_pool_config()

GPU_IDS   = POOL_CFG['gpu_ids']
N_WORKERS = POOL_CFG['n_workers']
INTRA     = POOL_CFG['intra_op_threads']
INTER     = POOL_CFG['inter_op_threads']
MODE      = POOL_CFG['mode']

print(f'mode      : {MODE}')
print(f'workers   : {N_WORKERS}')
print(f'gpu_ids   : {GPU_IDS if GPU_IDS else "(CPU only)"}')
print(f'tf threads: intra={INTRA}, inter={INTER}')


mode      : GPU
workers   : 2
gpu_ids   : [0, 1]
tf threads: intra=1, inter=1


## Build the worker pool

Each worker pulls one entry off `gpu_queue` exactly once at startup
(loky's `initializer`) and pins `CUDA_VISIBLE_DEVICES` before TF sees a
GPU. In CPU mode the entry is `None`, which triggers
`tf.config.set_visible_devices([], 'GPU')` so TF can't accidentally find
the GPU through another path. `reuse=False` forces a fresh pool if you
re-run this cell after changing the config.


In [3]:
mgr = mp.Manager()
gpu_queue = mgr.Queue()

slots = list(GPU_IDS) if GPU_IDS else [None] * N_WORKERS
assert len(slots) == N_WORKERS, 'one queue slot per worker'
for g in slots:
    gpu_queue.put(g)

executor = get_reusable_executor(
    max_workers=N_WORKERS,
    initializer=init_worker,
    initargs=(gpu_queue, INTRA, INTER),
    reuse=False,
)
print(f'pool ready: {N_WORKERS} workers, slots={slots}')


pool ready: 2 workers, slots=[0, 1]


## Sanity check: what do the workers actually see?

Submit one cheap probe per worker and confirm that:

- **GPU mode**: each worker reports a different `CUDA_VISIBLE_DEVICES`
  and a single GPU device.
- **CPU mode**: every worker reports `CUDA_VISIBLE_DEVICES = '-1'` and
  an empty `gpu_devices` list.

If a row shows the wrong state, the pool's pinning didn't take effect —
re-run the pool cell to get a fresh pool.


In [4]:
def _check():
    import os, tensorflow as tf
    return {
        'pid': os.getpid(),
        'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES'),
        'gpu_devices': [d.name for d in tf.config.list_physical_devices('GPU')],
        'intra_threads': tf.config.threading.get_intra_op_parallelism_threads(),
        'inter_threads': tf.config.threading.get_inter_op_parallelism_threads(),
    }

# Submit several so loky has a reason to spin up every worker.
checks = [executor.submit(_check) for _ in range(N_WORKERS * 4)]
df = pd.DataFrame([c.result() for c in checks]).drop_duplicates(subset='pid').reset_index(drop=True)
print(f'{len(df)} unique workers (expected {N_WORKERS})')
df


E0000 00:00:1777172573.445080 2153020 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777172573.445077 2153021 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777172573.449000 2153020 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1777172573.449347 2153021 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777172573.461107 2153020 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172573.461112 2153021 computation_placer.cc:177] computation placer already registered. Please check li

2 unique workers (expected 2)


,pid,CUDA_VISIBLE_DEVICES,gpu_devices,intra_threads,inter_threads
0,2153021,0,[/physical_device:GPU:0],1,1
1,2153020,1,[/physical_device:GPU:0],1,1


## Build the job list

`KFold` splits are generated in the parent (deterministic, cheap) and the
fold slices are passed to workers as numpy arrays. Loky memmaps large
numpy arrays automatically, so this is fast even for the bigger datasets.


In [5]:
DATASETS = ['SJAFFE', 'Music']
jobs = []
for dataset_name in DATASETS:
    X, D = load_dataset(dataset_name, dir='dataset')
    print(f'{dataset_name}: {X.shape[0]} samples, {X.shape[1]} features, {D.shape[1]} labels')
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        Xtr, Xte = X[train_idx], X[test_idx]
        Dtr, Dte = D[train_idx], D[test_idx]
        for model_name in MODEL_NAMES:
            jobs.append((dataset_name, model_name, fold_idx, Xtr, Dtr, Xte, Dte))

total = len(jobs)
print(f'queued {total} jobs ({len(DATASETS)} datasets × {N_SPLITS} folds × {len(MODEL_NAMES)} models)')


SJAFFE: 213 samples, 243 features, 6 labels
Music: 360 samples, 128 features, 9 labels
queued 160 jobs (2 datasets × 10 folds × 8 models)


## Submit + collect

Submission is non-blocking; results stream back via `as_completed` so the
log shows progress as folds finish. Per-fold failures are caught and
printed but don't stop the run.


In [6]:
futures = {
    executor.submit(run_one_fold, ds, m, fi, Xtr, Dtr, Xte, Dte, N_EPOCHS): (ds, m, fi)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte) in jobs
}

raw_results = []
failures = []
for i, fut in enumerate(as_completed(futures), start=1):
    ds, m, fi = futures[fut]
    try:
        raw_results.append(fut.result())
        status = 'ok'
    except Exception as e:
        failures.append((ds, m, fi, repr(e)))
        status = f'FAILED ({type(e).__name__}: {e})'
    print(f'[{i:4d}/{total}] {ds:12s} | fold {fi:2d} | {m:30s} {status}')

print(f'\ndone: {len(raw_results)} ok, {len(failures)} failed')


I0000 00:00:1777172589.961729 2153021 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8074 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3 MIG 1g.10gb, pci bus id: 0000:06:00.0, compute capability: 9.0
I0000 00:00:1777172589.961739 2153020 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8074 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3 MIG 1g.10gb, pci bus id: 0000:0c:00.0, compute capability: 9.0


[   1/160] SJAFFE       | fold  1 | EDL_LDL (loglikelihood)        ok
[   2/160] SJAFFE       | fold  1 | EDL_LDL (bayes_mse)            ok


E0000 00:00:1777172622.046400 2154123 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777172622.050798 2154123 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777172622.062980 2154123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172622.063079 2154123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172622.063111 2154123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777172622.063135 2154123 computation_placer.cc:177] computation placer already registered. Please check linka

[   3/160] SJAFFE       | fold  1 | BEDL_LDL (bayes_mse)           ok
[   4/160] SJAFFE       | fold  1 | BEDL_LDL (loglikelihood)       ok
[   5/160] SJAFFE       | fold  1 | AA_BP                          ok
[   6/160] SJAFFE       | fold  1 | Duo_LDL                        ok
[   7/160] SJAFFE       | fold  1 | SA_BFGS                        ok
[   8/160] SJAFFE       | fold  2 | EDL_LDL (loglikelihood)        ok
[   9/160] SJAFFE       | fold  1 | SNEFY_LDL                      ok
[  10/160] SJAFFE       | fold  2 | EDL_LDL (bayes_mse)            ok
[  11/160] SJAFFE       | fold  2 | BEDL_LDL (loglikelihood)       ok
[  12/160] SJAFFE       | fold  2 | BEDL_LDL (bayes_mse)           ok
[  13/160] SJAFFE       | fold  2 | AA_BP                          ok
[  14/160] SJAFFE       | fold  2 | Duo_LDL                        ok
[  15/160] SJAFFE       | fold  2 | SA_BFGS                        ok
[  16/160] SJAFFE       | fold  3 | EDL_LDL (loglikelihood)        ok
[  17/160] SJAFFE   

In [7]:
import os, psutil

# Pool's worker PIDs — loky stashes them on the executor.
worker_pids = list(executor._processes.keys())
alive = []
dead = []
for pid in worker_pids:
    try:
        p = psutil.Process(pid)
        if p.is_running() and p.status() != psutil.STATUS_ZOMBIE:
            alive.append((pid, p.status(), p.memory_info().rss / 1e9, p.cpu_percent(interval=0.5)))
        else:
            dead.append(pid)
    except psutil.NoSuchProcess:
        dead.append(pid)

print(f'alive workers ({len(alive)}/{N_WORKERS}):')
for pid, status, rss_gb, cpu in alive:
    print(f'  pid={pid}  status={status}  rss={rss_gb:.1f}GB  cpu={cpu:.0f}%')
print(f'dead workers: {dead}')

print(f'\\nresults so far: {len(raw_results)} / {total}')
print(f'completed futures: {sum(f.done() for f in futures)}')
print(f'pending futures:   {sum(not f.done() for f in futures)}')


alive workers (2/2):
  pid=2154123  status=sleeping  rss=1.2GB  cpu=0%
  pid=2154130  status=sleeping  rss=1.2GB  cpu=0%
dead workers: []
\nresults so far: 151 / 160
completed futures: 160
pending futures:   0


## Bucket results into per-model DataFrames

`per_model_results[(dataset, model_name)]` is a DataFrame with one row per
fold; columns are the recorded metrics.


In [8]:
buckets = defaultdict(list)
for r in raw_results:
    buckets[(r['dataset'], r['model'])].append(r['scores'])

per_model_results = {key: pd.DataFrame(rows) for key, rows in buckets.items()}
print(f'{len(per_model_results)} (dataset, model) combinations have results')


16 (dataset, model) combinations have results


## Per-model fold tables

Inspect any single (dataset, model) DataFrame:


In [9]:
# Pick the first (dataset, model) that actually has results so this cell
# doesn't KeyError if you ran with a reduced DATASETS list.
if per_model_results:
    first_key = next(iter(per_model_results))
    print(f'showing: {first_key}')
    display(per_model_results[first_key])
else:
    print('no results yet — run the submit cell first')


showing: ('SJAFFE', 'EDL_LDL (loglikelihood)')


,chebyshev,clark,canberra,kl_divergence,cosine,intersection,mean_uncertainty,uncertainty_calibration
0,0.138607,0.472805,1.018608,0.095152,0.910793,0.823833,0.857143,0.165713
1,0.126762,0.447226,0.949772,0.078298,0.925512,0.837772,0.857143,0.102081
2,0.128733,0.433697,0.895711,0.079212,0.925282,0.845774,0.857143,-0.074767
3,0.114381,0.388257,0.802849,0.062386,0.939763,0.862132,0.857143,-0.227921
4,0.100869,0.409850,0.857795,0.059956,0.945069,0.859144,0.857143,-0.026099
5,0.125406,0.426507,0.899383,0.082887,0.923913,0.845752,0.857143,-0.331271
6,0.119596,0.416328,0.874798,0.074867,0.930406,0.850723,0.857143,0.136822
7,0.110998,0.416299,0.862358,0.064974,0.939349,0.856253,0.857143,0.220193
8,0.111769,0.408368,0.859427,0.066510,0.937381,0.854850,0.857143,0.378394
9,0.117281,0.391949,0.837920,0.070193,0.933585,0.855037,0.857143,0.242082


## Combined summary — mean ± std across folds

One row per (dataset, model); columns are `metric_mean` / `metric_std`.


In [10]:
def summarize(df):
    out = {}
    for col in df.columns:
        out[f'{col}_mean'] = df[col].mean()
        out[f'{col}_std']  = df[col].std()
    return out


summary_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name, **summarize(df)}
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index(['dataset', 'model'])
summary


chebyshev_mean  chebyshev_std  clark_mean  \
dataset model                                                                 
SJAFFE  EDL_LDL (loglikelihood)         0.119440       0.010792    0.421129   
        EDL_LDL (bayes_mse)             0.119322       0.010363    0.426512   
        BEDL_LDL (bayes_mse)            0.119828       0.010457    0.429630   
        BEDL_LDL (loglikelihood)        0.119426       0.011207    0.419076   
        AA_BP                           0.189066       0.057742    0.709761   
        Duo_LDL                         0.119176       0.010422    0.424920   
        SA_BFGS                         0.089831       0.008980    0.348005   
        SNEFY_LDL                       0.342765       0.176949    1.356375   
Music   EDL_LDL (loglikelihood)         0.079003       0.004350    0.771701   
        EDL_LDL (bayes_mse)             0.077473       0.004577    0.771727   
        BEDL_LDL (loglikelihood)        0.077095       0.005997    0.725816   
        BEDL_LDL (bayes_mse)            0.076034       0.006146    0.724625   
        AA_BP                           0.205648       0.114962    1.122499   
        Duo_LDL                         0.073137       0.007139    0.715091   
        SA_BFGS                         0.098572       0.007803    0.859496   
        SNEFY_LDL                       0.107468            NaN    1.287714   

                                  clark_std  canberra_mean  canberra_std  \
dataset model                                                              
SJAFFE  EDL_LDL (loglikelihood)    0.025401       0.885862      0.060861   
        EDL_LDL (bayes_mse)        0.025296       0.888891      0.058743   
        BEDL_LDL (bayes_mse)       0.025342       0.899155      0.059641   
        BEDL_LDL (loglikelihood)   0.025576       0.882482      0.060566   
        AA_BP                      0.160399       1.495359      0.361597   
        Duo_LDL                    0.025284       0.886697      0.058991   
        SA_BFGS                    0.034008       0.707695      0.080484   
        SNEFY_LDL                  0.317620       2.884027      0.836238   
Music   EDL_LDL (loglikelihood)    0.045767       1.943885      0.138128   
        EDL_LDL (bayes_mse)        0.046101       1.943715      0.138270   
        BEDL_LDL (loglikelihood)   0.059062       1.837876      0.157822   
        BEDL_LDL (bayes_mse)       0.058553       1.829708      0.162537   
        AA_BP                      0.212352       2.925621      0.614755   
        Duo_LDL                    0.064195       1.811366      0.177207   
        SA_BFGS                    0.054782       2.226006      0.158546   
        SNEFY_LDL                       NaN       2.983426           NaN   

                                  kl_divergence_mean  kl_divergence_std  \
dataset model                                                             
SJAFFE  EDL_LDL (loglikelihood)             0.073443           0.010821   
        EDL_LDL (bayes_mse)                 0.073372           0.009659   
        BEDL_LDL (bayes_mse)                0.074121           0.009305   
        BEDL_LDL (loglikelihood)            0.074175           0.012019   
        AA_BP                               0.211559           0.099434   
        Duo_LDL                             0.073010           0.009717   
        SA_BFGS                             0.046651           0.009437   
        SNEFY_LDL                           0.858193           0.421661   
Music   EDL_LDL (loglikelihood)             0.116689           0.013323   
        EDL_LDL (bayes_mse)                 0.116442           0.013537   
        BEDL_LDL (loglikelihood)            0.109839           0.018890   
        BEDL_LDL (bayes_mse)                0.108859           0.017828   
        AA_BP                               0.348238           0.176484   
        Duo_LDL                             0.104814           0.018686   
        SA_BFGS                           

### Compact view: `mean ± std` per metric


In [11]:
def fmt(mean, std):
    if pd.isna(mean):
        return ''
    return f'{mean:.4f} ± {std:.4f}'


compact_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name}
    for col in df.columns:
        row[col] = fmt(df[col].mean(), df[col].std())
    compact_rows.append(row)

compact = pd.DataFrame(compact_rows).set_index(['dataset', 'model'])
compact


chebyshev            clark  \
dataset model                                                        
SJAFFE  EDL_LDL (loglikelihood)   0.1194 ± 0.0108  0.4211 ± 0.0254   
        EDL_LDL (bayes_mse)       0.1193 ± 0.0104  0.4265 ± 0.0253   
        BEDL_LDL (bayes_mse)      0.1198 ± 0.0105  0.4296 ± 0.0253   
        BEDL_LDL (loglikelihood)  0.1194 ± 0.0112  0.4191 ± 0.0256   
        AA_BP                     0.1891 ± 0.0577  0.7098 ± 0.1604   
        Duo_LDL                   0.1192 ± 0.0104  0.4249 ± 0.0253   
        SA_BFGS                   0.0898 ± 0.0090  0.3480 ± 0.0340   
        SNEFY_LDL                 0.3428 ± 0.1769  1.3564 ± 0.3176   
Music   EDL_LDL (loglikelihood)   0.0790 ± 0.0043  0.7717 ± 0.0458   
        EDL_LDL (bayes_mse)       0.0775 ± 0.0046  0.7717 ± 0.0461   
        BEDL_LDL (loglikelihood)  0.0771 ± 0.0060  0.7258 ± 0.0591   
        BEDL_LDL (bayes_mse)      0.0760 ± 0.0061  0.7246 ± 0.0586   
        AA_BP                     0.2056 ± 0.1150  1.1225 ± 0.2124   
        Duo_LDL                   0.0731 ± 0.0071  0.7151 ± 0.0642   
        SA_BFGS                   0.0986 ± 0.0078  0.8595 ± 0.0548   
        SNEFY_LDL                    0.1075 ± nan     1.2877 ± nan   

                                         canberra    kl_divergence  \
dataset model                                                        
SJAFFE  EDL_LDL (loglikelihood)   0.8859 ± 0.0609  0.0734 ± 0.0108   
        EDL_LDL (bayes_mse)       0.8889 ± 0.0587  0.0734 ± 0.0097   
        BEDL_LDL (bayes_mse)      0.8992 ± 0.0596  0.0741 ± 0.0093   
        BEDL_LDL (loglikelihood)  0.8825 ± 0.0606  0.0742 ± 0.0120   
        AA_BP                     1.4954 ± 0.3616  0.2116 ± 0.0994   
        Duo_LDL                   0.8867 ± 0.0590  0.0730 ± 0.0097   
        SA_BFGS                   0.7077 ± 0.0805  0.0467 ± 0.0094   
        SNEFY_LDL                 2.8840 ± 0.8362  0.8582 ± 0.4217   
Music   EDL_LDL (loglikelihood)   1.9439 ± 0.1381  0.1167 ± 0.0133   
        EDL_LDL (bayes_mse)       1.9437 ± 0.1383  0.1164 ± 0.0135   
        BEDL_LDL (loglikelihood)  1.8379 ± 0.1578  0.1098 ± 0.0189   
        BEDL_LDL (bayes_mse)      1.8297 ± 0.1625  0.1089 ± 0.0178   
        AA_BP                     2.9256 ± 0.6148  0.3482 ± 0.1765   
        Duo_LDL                   1.8114 ± 0.1772  0.1048 ± 0.0187   
        SA_BFGS                   2.2260 ± 0.1585  0.1832 ± 0.0301   
        SNEFY_LDL                    2.9834 ± nan     0.3529 ± nan   

                                           cosine     intersection  \
dataset model                                                        
SJAFFE  EDL_LDL (loglikelihood)   0.9311 ± 0.0101  0.8491 ± 0.0115   
        EDL_LDL (bayes_mse)       0.9309 ± 0.0090  0.8485 ± 0.0107   
        BEDL_LDL (bayes_mse)      0.9302 ± 0.0088  0.8468 ± 0.0110   
        BEDL_LDL (loglikelihood)  0.9306 ± 0.0111  0.8496 ± 0.0117   
        AA_BP                     0.8271 ± 0.0707  0.7422 ± 0.0652   
        Duo_LDL                   0.9313 ± 0.0091  0.8489 ± 0.0108   
        SA_BFGS                   0.9570 ± 0.0082  0.8820 ± 0.0130   
        SNEFY_LDL                 0.6579 ± 0.1429  0.5618 ± 0.1462   
Music   EDL_LDL (loglikelihood)   0.9140 ± 0.0093  0.8052 ± 0.0143   
        EDL_LDL (bayes_mse)       0.9145 ± 0.0095  0.8054 ± 0.0145   
        BEDL_LDL (loglikelihood)  0.9188 ± 0.0139  0.8171 ± 0.0169   
        BEDL_LDL (bayes_mse)      0.9192 ± 0.0131  0.8175 ± 0.0174   
        AA_BP                     0.7478 ± 0.1112  0.6701 ± 0.0881   
        Duo_LDL                   0.9229 ± 0.0138  0.8202 ± 0.0190   
        SA_BFGS                   0.8711 ± 0.0198  0.7722 ± 0.0191   
        SNEFY_LDL                    0.8575 ± nan     0.7457 ± nan   

                                 mean_uncertainty uncertainty_calibration  
dataset model                                                              
SJAFFE  EDL_LDL (loglikelihood)   0.8571 ± 0.0000         0.0585 ± 0.2213  
        EDL_LDL (bayes_mse)  

## Uncertainty results (EDL_LDL, BEDL_LDL, SNEFY_LDL only)

- `mean_uncertainty` — average per-sample uncertainty on test (model-specific
  scale; lower = more confident).
- `uncertainty_calibration` — Spearman ρ between per-sample uncertainty and
  per-sample KL divergence error. Higher = uncertainty better predicts error.


In [15]:
uncertainty_models = {
    'EDL_LDL (loglikelihood)', 'EDL_LDL (bayes_mse)',
    'BEDL_LDL (loglikelihood)', 'BEDL_LDL (bayes_mse)',
    'SNEFY_LDL',
}

uncertainty_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if model_name not in uncertainty_models or df.empty:
        continue
    if 'mean_uncertainty' not in df.columns:
        continue
    uncertainty_rows.append({
        'dataset': dataset_name,
        'model': model_name,
        'mean_uncertainty':        fmt(df['mean_uncertainty'].mean(),        df['mean_uncertainty'].std()),
        'uncertainty_calibration': fmt(df['uncertainty_calibration'].mean(), df['uncertainty_calibration'].std()),
    })

uncertainty_summary = pd.DataFrame(uncertainty_rows).set_index(['dataset', 'model'])
uncertainty_summary


mean_uncertainty uncertainty_calibration
dataset model                                                            
SJAFFE  EDL_LDL (loglikelihood)   0.8571 ± 0.0000         0.1439 ± 0.0945
        EDL_LDL (bayes_mse)       0.8571 ± 0.0000        -0.0118 ± 0.0935
        BEDL_LDL (bayes_mse)      0.3852 ± 0.0011         0.1867 ± 0.2745
        BEDL_LDL (loglikelihood)  0.4027 ± 0.0050        -0.1086 ± 0.1315
        SNEFY_LDL                 0.2576 ± 0.1916        -0.0608 ± 0.2045

## Shut down the pool

Loky reuses pools by default; close it explicitly when you're done so the
worker processes (and any GPU memory they hold) are released.


In [16]:
executor.shutdown(wait=True, kill_workers=True)
